# Load Curated Data to Target Storage

This notebook stages processed outputs for Fabric or a local staging area, builds a load manifest, and produces Power BI-friendly dataset metadata.

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PurviewLoadFabric").getOrCreate()
root = Path.cwd().parent
processed_root = root / "data" / "processed"
reports_root = root / "data" / "reports"

reports_root.mkdir(parents=True, exist_ok=True)

target_root = os.getenv("FABRIC_ONELAKE_PATH") or os.getenv("FABRIC_LAKEHOUSE_PATH")
if target_root:
    target_root = Path(target_root)
    target_mode = "fabric_path"
else:
    target_root = reports_root / "fabric_staging"
    target_mode = "local_staging"

target_root.mkdir(parents=True, exist_ok=True)
print(json.dumps({"target_mode": target_mode, "target_root": str(target_root)}, indent=2))

In [ ]:
dataset_names = [
    "dim_collections",
    "dim_sources",
    "dim_assets",
    "dim_data_products",
    "fact_scans",
    "fact_quality",
    "relationships",
    "key_vault_validation",
]

loaded = {}
for dataset_name in dataset_names:
    source_path = processed_root / dataset_name
    if source_path.exists():
        loaded[dataset_name] = spark.read.parquet(str(source_path))
        print(f"Loaded {dataset_name} from {source_path}")
    else:
        print(f"Skipped {dataset_name}; no source data found")

if not loaded:
    raise RuntimeError("No processed datasets were found. Run notebooks 03 and 04 first.")

In [ ]:
manifest_items = []
for dataset_name, dataset_df in loaded.items():
    target_path = target_root / dataset_name
    dataset_df.write.mode("overwrite").parquet(str(target_path))
    manifest_items.append({
        "dataset": dataset_name,
        "record_count": dataset_df.count(),
        "target_path": str(target_path),
        "columns": dataset_df.columns,
    })
    print(f"Published {dataset_name} to {target_path}")

In [ ]:
load_manifest = {
    "generated_at": datetime.utcnow().isoformat(),
    "target_mode": target_mode,
    "target_root": str(target_root),
    "datasets": manifest_items,
}

powerbi_manifest = {
    "generated_at": datetime.utcnow().isoformat(),
    "recommended_tables": [item["dataset"] for item in manifest_items],
    "recommended_relationships": [
        "dim_collections.collection_id -> dim_sources.collection_id",
        "dim_collections.collection_id -> dim_assets.collection_id",
        "dim_data_products.data_product_id -> dim_assets.data_product_id",
        "dim_sources.source_id -> fact_scans.data_source_id",
        "dim_assets.asset_id -> fact_quality.asset_id",
    ],
}

manifest_path = reports_root / "load_manifest.json"
powerbi_path = reports_root / "powerbi_dataset_manifest.json"

with manifest_path.open("w", encoding="utf-8") as handle:
    json.dump(load_manifest, handle, indent=2)
with powerbi_path.open("w", encoding="utf-8") as handle:
    json.dump(powerbi_manifest, handle, indent=2)

print(json.dumps(load_manifest, indent=2))
print(f"Saved load manifest to {manifest_path}")
print(f"Saved Power BI dataset manifest to {powerbi_path}")